# System performance, small check on a T4: SmolVLM2-2.2B, one video, all its questions (Colab T4)

A quick run of `system_perf/` on a **T4** (16 GB, no bf16, so the model runs in **fp16**): SmolVLM2-2.2B only, the test
video `loc2_script1_seq2_rec1` with **all its 23 questions**, every method and condition of its budget-matched accuracy
run (run that notebook first: this one reads its `config_matched.yaml` and `summary.csv`). Per method and condition:
**TTFT** (vision side + prefill), **TPS** (64 greedy tokens), **VE / LD / overall peak dynamic memory**, **frames
encoded**, **vision-side GFLOPs**, **method overhead**, **energy and power** (NVML), decoder **memory vs batch size**
(1-8). One warm-up per condition (the A100 notebook uses 3). Details: `system_perf/README.md`.

Results go to `MyDrive/GazeVLM/system_perf_t4/smolvlm/` (separate from the A100 results). Accuracy in the report is
the A100 bf16 accuracy run's (4 test videos). Rough time on a T4: ~45 min.


In [ ]:
import os, sys, subprocess
import torch

# A small check on a T4 (fp16); any CUDA GPU works, the A100 notebook is the full protocol.
gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
print('GPU:', f'{gpu.name}, {gpu.total_memory / 1e9:.0f} GB' if gpu else 'none')
assert gpu, 'needs a GPU runtime: Runtime > Change runtime type > T4 GPU'

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
# num2words: needed by SmolVLM2's processor (install before transformers is first imported)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pytest', 'num2words', 'pyyaml', 'nvidia-ml-py'],
               check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
try:                                         # Hugging Face token from the Colab secret HF_TOKEN
    from google.colab import userdata
    from huggingface_hub import login
    if userdata.get('HF_TOKEN'):
        login(token=userdata.get('HF_TOKEN'))
except Exception as ex:
    print('no HF_TOKEN secret:', type(ex).__name__)

MODEL, LABEL = 'smolvlm', 'SmolVLM2-2.2B'
VIDEO = 'loc2_script1_seq2_rec1'     # a test video: 23 questions
ACC = '/content/drive/MyDrive/GazeVLM/accuracy_tables/smolvlm_matched'    # its budget-matched accuracy run
MATCHED = f'{ACC}/config_matched.yaml'
SYS = '/content/drive/MyDrive/GazeVLM/system_perf_t4'                    # separate from the A100 results
QUESTIONS, WARMUP, BATCH = 1000, 1, '1,2,4,8'                            # 1000 = all of the video's questions
assert os.path.exists(MATCHED), f'run the budget-matched SmolVLM2 accuracy notebook first ({MATCHED} missing)'
print('conditions from', MATCHED, '-> results in', f'{SYS}/{MODEL}')


In [ ]:
r = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_system_perf.py', 'tests/test_accuracy_tables.py', '-q'],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-2000:])
assert r.returncode == 0, 'unit tests failed'


## Measure

Parity first, then every row and condition of the matched config on the sampled questions. Each line printed is one
question: TTFT (vision side), TPS, VE / LD peak memory, frames encoded, visual tokens. Rerunning appends; the report
keeps the last record per question.


In [ ]:
get_ipython().system(f'python -m system_perf.measure --config {MATCHED} --model {MODEL} --videos {VIDEO} '
                     f'--questions {QUESTIONS} --warmup {WARMUP} --dtype auto --out-dir {SYS} --batch-sizes {BATCH}')


## Report

Mean +- s.d. per condition, speed-ups and memory savings vs Uniform K, method overheads, accuracy from the accuracy
run; `system_table.csv` and `system_table.tex` (rows for the desktop table of `system_performance.tex`).


In [ ]:
get_ipython().system(f'python -m system_perf.report --results {SYS} --model {MODEL} --accuracy {ACC}/summary.csv '
                     f'--label "{LABEL}"')
print(open(f'{SYS}/{MODEL}/system_table.tex').read())
